In [0]:
# Databricks notebook source

from __future__ import annotations

import uuid

# ============================================================
# Widgets
# ============================================================
dbutils.widgets.text("catalog", "phc_txdot")
catalog = dbutils.widgets.get("catalog").strip()

# ============================================================
# Constants
# ============================================================
SCRIPT_NAME = "440_build_gold_vw_dim_item_work_category.py"
RUN_ID = str(uuid.uuid4())

SOURCE_TABLE = f"{catalog}.gold.dim_item_work_category"
TARGET_VIEW = f"{catalog}.gold.vw_dim_item_work_category"
RUN_LOG_TABLE = f"{catalog}.staging.pipeline_run_log"

# ============================================================
# Helpers
# ============================================================
def sql_escape(value: str) -> str:
    return value.replace("'", "''")


def log_run(status: str, row_count: int | None, message: str) -> None:
    row_count_sql = "NULL" if row_count is None else str(row_count)
    spark.sql(f"""
        INSERT INTO {RUN_LOG_TABLE}
        VALUES (
              '{RUN_ID}'
            , '{SCRIPT_NAME}'
            , '{sql_escape(status)}'
            , {row_count_sql}
            , '{sql_escape(message)}'
            , current_timestamp()
        )
    """)

# ============================================================
# Build gold.vw_dim_item_work_category
# ============================================================
try:
    spark.sql(f"DROP VIEW IF EXISTS {TARGET_VIEW}")

    spark.sql(f"""
    CREATE VIEW {TARGET_VIEW} AS
    WITH ranked AS (
        SELECT
              item_work_category_key
            , effective_specification_description
            , specification_description
            , item_work_category
            , item_work_category_source
            , is_defaulted_work_category
            , ROW_NUMBER() OVER (
                PARTITION BY item_work_category_key
                ORDER BY
                      is_defaulted_work_category ASC
                    , item_work_category_source
                    , effective_specification_description
              ) AS rn
        FROM {SOURCE_TABLE}
    )
    SELECT
          item_work_category_key              AS `Item Work Category Key`
        , effective_specification_description AS `Effective Specification Description`
        , specification_description           AS `Specification Description`
        , item_work_category                  AS `Item Work Category`
        , item_work_category_source           AS `Item Work Category Source`
        , is_defaulted_work_category          AS `Is Defaulted Work Category`
    FROM ranked
    WHERE rn = 1
    """)

    row_count = spark.sql(f"""
        SELECT COUNT(*) AS row_count
        FROM {TARGET_VIEW}
    """).collect()[0]["row_count"]

    log_run("SUCCESS", row_count, "Built gold.vw_dim_item_work_category successfully.")

    print("=" * 70)
    print("Built gold.vw_dim_item_work_category")
    print(f"Row count: {row_count:,}")
    print("=" * 70)

except Exception as exc:
    log_run("FAILED", None, str(exc))
    raise